In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [6]:
from typing import Any

from langchain.agents.middleware import AgentMiddleware, AgentState, hook_config
from langgraph.runtime import Runtime

class ContentFilterMiddleware(AgentMiddleware):
    """Deterministic guardrail: Block requests containing banned keywords."""

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        # Get the first user message
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        # Check for banned keywords
        for keyword in self.banned_keywords:
            if keyword in content:
                # Block execution before any processing
                return {
                    "messages": [{
                        "role": "assistant",
                        "content": "부적절한 콘텐츠가 포함된 요청은 처리할 수 없습니다. 요청을 다시 작성해 주세요."
                    }],
                    "jump_to": "end"
                }

        return None

In [3]:
# Use the custom guardrail
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    # tools=[search_tool, calculator_tool],
    middleware=[
        ContentFilterMiddleware(
            banned_keywords=["해킹", "hack", "exploit", "malware"]
        ),
    ],
)

In [7]:
# This request will be blocked before any processing
result = agent.invoke({
    "messages": [{"role": "user", "content": "데이터베이스를 해킹하려면 어떻게 해야 하나요?"}]
})

In [8]:
result

{'messages': [HumanMessage(content='데이터베이스를 해킹하려면 어떻게 해야 하나요?', additional_kwargs={}, response_metadata={}, id='f40d7f4e-0104-4d2f-bb64-66fbce4b3995'),
  AIMessage(content='I cannot process requests containing inappropriate content. Please rephrase your request.', additional_kwargs={}, response_metadata={}, id='965c56ed-9351-4b3b-8857-71194e69cf8a', tool_calls=[], invalid_tool_calls=[])]}